# Eco-Router-Ai Starter Notebook

This notebook follows the real app flow from [app/main.py](../app/main.py):

- detect the repository root
- load local or server-side environment variables
- inspect the sample input tasks
- classify and route prompts with the repo's rule-based logic
- optionally run the full pipeline and preview the written results

In [19]:
from pathlib import Path
import json
import os
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'app' / 'main.py').exists() or (candidate / 'README.md').exists():
            return candidate
    return start


repo_root = find_repo_root(Path.cwd().resolve())
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

input_path = repo_root / 'input' / 'tasks.json'
output_dir = repo_root / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Repository root: {repo_root}')
print(f'Working directory: {Path.cwd()}')
print(f'Input tasks: {input_path}')
print(f'Output directory: {output_dir}')

Repository root: D:\kampf\projects\EcoRouteAi\Eco-Router-Ai
Working directory: D:\kampf\projects\EcoRouteAi\Eco-Router-Ai
Input tasks: D:\kampf\projects\EcoRouteAi\Eco-Router-Ai\input\tasks.json
Output directory: D:\kampf\projects\EcoRouteAi\Eco-Router-Ai\output


In [20]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

repo_base = globals().get('repo_root', Path.cwd())
env_file = Path(os.environ.get('ECO_ROUTER_ENV_FILE', str(repo_base / '.env')))
required_vars = ['FIREWORKS_API_KEY', 'FIREWORKS_BASE_URL', 'ALLOWED_MODELS']

loaded_env = False
if load_dotenv is not None and env_file.exists():
    load_dotenv(env_file)
    loaded_env = True

print(f'.env file: {env_file}')
print(f'Loaded .env: {loaded_env}')
missing = [name for name in required_vars if not os.getenv(name)]
if missing:
    print('Missing runtime variables:', ', '.join(missing))
else:
    print('All required runtime variables are present')

print('INPUT_PATH =', os.environ.get('INPUT_PATH', '/input/tasks.json'))
print('OUTPUT_PATH =', os.environ.get('OUTPUT_PATH', '/output/results.json'))
print('ALLOWED_MODELS =', os.environ.get('ALLOWED_MODELS', '<not set>'))

.env file: D:\kampf\projects\EcoRouteAi\Eco-Router-Ai\.env
Loaded .env: True
All required runtime variables are present
INPUT_PATH = ./input/tasks.json
OUTPUT_PATH = ./output/results.json
ALLOWED_MODELS = minimax-m3


In [21]:
from app.io.reader import load_tasks

tasks = load_tasks(str(input_path))
print(f'Loaded {len(tasks)} task(s) from {input_path}')
for task in tasks:
    preview = (task.prompt or '').strip().replace('\n', ' ')
    suffix = '...' if len(preview) > 120 else ''
    print(f'- {task.task_id}: {preview[:120]}{suffix}')

Loaded 3 task(s) from D:\kampf\projects\EcoRouteAi\Eco-Router-Ai\input\tasks.json
- t1: Summarise the following text in one sentence: The quick brown fox jumps over the lazy dog.
- t2: Write a Python function that reverses a string. Include type hints.
- t3: Calculate the percentage increase from 50 to 75 and show the steps.


In [22]:
from app.router.classifier import classify_task
from app.router.model_selector import select_model

allowed_models = [
    model.strip()
    for model in os.environ.get('ALLOWED_MODELS', 'minimax-m3,kimi-k2p7-code,gemma-4-31b-it').split(',')
    if model.strip()
 ]

routing_rows = []
for task in tasks:
    classified = classify_task(task.model_dump())
    chosen_model, rationale = select_model(classified['classification'], allowed_models)
    routing_rows.append({
        'task_id': task.task_id,
        'category': classified['classification'].get('category'),
        'confidence': classified['classification'].get('confidence'),
        'chosen_model': chosen_model,
        'reason': rationale.get('reason'),
    })

print(json.dumps(routing_rows, indent=2))

[
  {
    "task_id": "t1",
    "category": "text_summarisation",
    "confidence": 0.9,
    "chosen_model": "minimax-m3",
    "reason": "high_confidence_choose_smallest"
  },
  {
    "task_id": "t2",
    "category": "factual_knowledge",
    "confidence": 0.4,
    "chosen_model": "minimax-m3",
    "reason": "medium_confidence"
  },
  {
    "task_id": "t3",
    "category": "mathematical_reasoning",
    "confidence": 0.9,
    "chosen_model": "minimax-m3",
    "reason": "high_confidence_choose_smallest"
  }
]


In [23]:
from time import perf_counter

from app.clients.fireworks_client import call_chat_model, extract_message_text
from app.core.config import load_settings
from app.io.writer import write_results
from app.models.result import Result
from app.router.classifier import classify_task
from app.router.model_selector import select_model


def response_usage(response):
    usage = response.get('usage') if isinstance(response, dict) else None
    if not isinstance(usage, dict):
        return {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}

    prompt_tokens = int(usage.get('prompt_tokens') or usage.get('input_tokens') or 0)
    completion_tokens = int(usage.get('completion_tokens') or usage.get('output_tokens') or 0)
    total_tokens = int(usage.get('total_tokens') or (prompt_tokens + completion_tokens))
    return {
        'prompt_tokens': prompt_tokens,
        'completion_tokens': completion_tokens,
        'total_tokens': total_tokens,
    }


settings = load_settings()
allowed_models_str = ', '.join(settings.allowed_models)

print('Resolved settings:')
print(f'- input_path: {settings.input_path}')
print(f'- output_path: {settings.output_path}')
print(f'- allowed_models: {allowed_models_str}')

if os.environ.get('RUN_FULL_PIPELINE', '1') == '1':
    analytics_rows = []
    results = []
    pipeline_started = perf_counter()

    for task in tasks:
        routed = classify_task(task.model_dump())
        chosen_model, rationale = select_model(routed['classification'], settings.allowed_models)

        model_elapsed_seconds = 0.0
        usage = {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}
        answer_text = ''

        if chosen_model:
            call_started = perf_counter()
            response = call_chat_model(
                task.prompt,
                chosen_model,
                base_url=settings.fireworks_base_url,
                api_key=settings.fireworks_api_key,
                timeout=settings.request_timeout,
            )
            model_elapsed_seconds = perf_counter() - call_started
            if response:
                answer_text = extract_message_text(response) or ''
                usage = response_usage(response)
            else:
                print(f'Warning: no response for task {task.task_id} using model {chosen_model}')
        else:
            print(f'Warning: no model chosen for task {task.task_id} (classification={routed["classification"]})')

        results.append(Result(task_id=task.task_id, answer=answer_text))
        analytics_rows.append({
            'task_id': task.task_id,
            'category': routed['classification'].get('category'),
            'confidence': routed['classification'].get('confidence'),
            'chosen_model': chosen_model,
            'reason': rationale.get('reason'),
            'model_elapsed_seconds': round(model_elapsed_seconds, 4),
            **usage,
        })

    write_started = perf_counter()
    write_results(results, settings.output_path)
    write_elapsed_seconds = perf_counter() - write_started
    pipeline_elapsed_seconds = perf_counter() - pipeline_started

    summary = {
        'tasks': len(results),
        'pipeline_elapsed_seconds': round(pipeline_elapsed_seconds, 4),
        'write_elapsed_seconds': round(write_elapsed_seconds, 4),
        'prompt_tokens': sum(row['prompt_tokens'] for row in analytics_rows),
        'completion_tokens': sum(row['completion_tokens'] for row in analytics_rows),
        'total_tokens': sum(row['total_tokens'] for row in analytics_rows),
        'average_model_elapsed_seconds': round(
            sum(row['model_elapsed_seconds'] for row in analytics_rows) / len(analytics_rows),
            4,
        ) if analytics_rows else 0.0,
    }

    print('Analytics summary:')
    print(json.dumps(summary, indent=2))
    print('Per-task analytics:')
    print(json.dumps(analytics_rows, indent=2))

    results_path = Path(settings.output_path)
    if results_path.exists():
        print('Results file:')
        print(results_path.read_text())
    else:
        print(f'No results file found at {results_path}')
else:
    print('Set RUN_FULL_PIPELINE=1 to execute the full app pipeline from this notebook.')
    print('The notebook is ready for inspection, routing demos, and environment validation without running the networked step.')

Resolved settings:
- input_path: ./input/tasks.json
- output_path: ./output/results.json
- allowed_models: minimax-m3
Analytics summary:
{
  "tasks": 3,
  "pipeline_elapsed_seconds": 1.7457,
  "write_elapsed_seconds": 0.0011,
  "prompt_tokens": 0,
  "completion_tokens": 0,
  "total_tokens": 0,
  "average_model_elapsed_seconds": 0.4301
}
Per-task analytics:
[
  {
    "task_id": "t1",
    "category": "text_summarisation",
    "confidence": 0.9,
    "chosen_model": "minimax-m3",
    "reason": "high_confidence_choose_smallest",
    "model_elapsed_seconds": 0.4711,
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "total_tokens": 0
  },
  {
    "task_id": "t2",
    "category": "factual_knowledge",
    "confidence": 0.4,
    "chosen_model": "minimax-m3",
    "reason": "medium_confidence",
    "model_elapsed_seconds": 0.4165,
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "total_tokens": 0
  },
  {
    "task_id": "t3",
    "category": "mathematical_reasoning",
    "confidence":